# Policies and values

_Artificial Intelligence (CS221)_

**A policy is your game plan. Its value is how much reward that plan earns on average.**

A policy tells the agent what to do: in each state, pick this action.

     The value of a policy at a state is the average total reward you collect by following it from there.

---

This notebook works through policy evaluation one step at a time — turning a fixed policy into a Markov chain, then solving the Bellman equation to find what each state is worth. Run each cell, read the short note above it, and let the link between rewards and long-run value sink in. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

Once you fix a policy, the actions are decided, so the MDP collapses into a plain **Markov chain**. We evaluate that policy in three steps: (1) write down the chain and rewards, (2) solve the Bellman equation for the values, (3) check the values satisfy that equation.

### Step 1 — A fixed policy as a Markov chain

Because the policy already chose every action, all that remains is `P[s, s']`, the probability of moving from state `s` to state `s'` under that policy. State `2` is **absorbing** — its row sends all probability back to itself, so once you arrive you stay. The vector `r` is the immediate reward collected in each state, and `gamma` discounts future rewards relative to present ones.

In [ ]:
# A fixed policy induces a Markov chain. P[s, s'] = transition under the policy.
P = np.array([[0.5, 0.5, 0.0],
              [0.0, 0.5, 0.5],
              [0.0, 0.0, 1.0]])   # state 2 is absorbing

# Immediate reward collected in each state.
r = np.array([0.0, 1.0, 10.0])

# Discount factor on future reward.
gamma = 0.9

### Step 2 — Solve the Bellman equation for the values

The value `V(s)` is the expected discounted reward from following the policy starting at `s`. It satisfies `V = r + gamma * P @ V`: today's reward plus the discounted value of wherever you go next. Rearranging gives the linear system `(I - gamma * P) V = r`, which `np.linalg.solve` solves exactly — no iteration needed, because the policy is fixed.

In [ ]:
# Policy evaluation: V = r + gamma P V  =>  (I - gamma P) V = r.
I = np.eye(3)
A = I - gamma * P
V = np.linalg.solve(A, r)

print("policy values V(s):", np.round(V, 3))

### Step 3 — Verify the Bellman fixed point

If `V` is correct, plugging it back into the right-hand side `r + gamma * P @ V` must reproduce `V` exactly. This re-applies one Bellman backup and confirms nothing moves — a quick proof that we found the true policy values rather than an approximation.

In [ ]:
# Re-apply the Bellman equation; this should reproduce V exactly.
bellman_rhs = r + gamma * P @ V

print("Bellman check    :", np.round(bellman_rhs, 3))

## Visualize the data & results

_If the warehouse robot always advances down the aisle, what is each spot worth?_

We evaluate the fixed `always advance` policy on a warehouse aisle, where each step has a small cost and reaching the shelf pays off, then draw the resulting state values as a heatmap.

### Step 1 — Evaluate the 'always advance' policy

The three states are positions along the aisle; the policy always advances, so `P` describes the (slightly slippery) forward motion. Each step carries a small cost of `-0.1`, while reaching the shelf is worth `10`. Solving `(I - gamma P) V = r` again gives the value of every spot under this policy.

In [ ]:
# Fixed policy 'always advance' transition matrix.
P = np.array([[0.8, 0.2, 0.0],
              [0.0, 0.7, 0.3],
              [0.0, 0.0, 1.0]])

# Small step cost everywhere; big payoff at the shelf.
r = np.array([-0.1, -0.1, 10.0])
gamma = 0.9

# Solve (I - gamma P) V = r for the policy values.
V = np.linalg.solve(np.eye(3) - gamma * P, r)

### Step 2 — Draw the value of each spot

The heatmap shows `V(s)` across the aisle. Spots closer to the shelf are worth more because the rewarding state is fewer (discounted) steps away. This is exactly what policy evaluation buys us: a single number per state summarizing the long-run payoff of sticking with the policy from there.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.5))

# Show the value vector as a single-row heatmap.
im = ax.imshow(V.reshape(1, 3), cmap="viridis", aspect="auto")

ax.set_yticks([0])
ax.set_yticklabels(["V"])
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["aisle start", "mid-aisle", "shelf"])

# Annotate each cell with its value.
for s in range(3):
    ax.text(s, 0, round(V[s], 2), ha="center", va="center", color="white")

ax.set_title("warehouse: policy values V(s) for always-advance")
fig.colorbar(im, ax=ax)
plt.show()